In [30]:
import requests
def get_youdosudoku_grid(difficulty="easy"):
    url = "https://www.youdosudoku.com/api/"
    
    # API şifresiz olduğu için sadece JSON verisi göndereceğimizi söylüyoruz. 
    # API key başlığını TAMAMEN sildik.
    headers = {
        "Content-Type": "application/json"
    }
    
    # Sadece zorluk derecesini gönderiyoruz, ekstra parametrelerle sunucuyu yormuyoruz.
    payload = {
        "difficulty": difficulty
    }
    
    try:
        # İstek atıyoruz
        response = requests.post(url, headers=headers, json=payload)
        
        # Eğer hala 400 alırsak sebebini tam olarak yazdırsın
        if response.status_code != 200:
            print(f"API Hatası ({response.status_code}):", response.text)
            return None
            
        data = response.json()
        
        # YouDoSudoku varsayılan olarak 'puzzle' anahtarında 81 karakterlik bir metin döner
        # Örn: "530070000600195000..."
        puzzle_string = data.get('puzzle', '')
        
        if not puzzle_string:
            print("Sudoku verisi alınamadı!")
            return None
            
        # 81 karakterlik o string'i alıp, senin algoritman için 9x9 listeye çeviriyoruz
        grid = []
        for i in range(9):
            satir = []
            for j in range(9):
                char = puzzle_string[i * 9 + j]
                # YouDoSudoku boşlukları '0' olarak gönderir, biz -1 yapıyoruz
                if char == '0':
                    satir.append(-1)
                else:
                    satir.append(int(char))
            grid.append(satir)
            
        return grid

    except Exception as e:
        print("Bağlantı sırasında bir hata oluştu:", e)
        return None

In [ ]:
#easy grid to test

sudoku_grid = [
    [ 5,  3, -1,   -1,  7, -1,   -1, -1, -1],
    [ 6, -1, -1,    1,  9,  5,   -1, -1, -1],
    [-1,  9,  8,   -1, -1, -1,   -1,  6, -1],

    [ 8, -1, -1,   -1,  6, -1,   -1, -1,  3],
    [ 4, -1, -1,    8, -1,  3,   -1, -1,  1],
    [ 7, -1, -1,   -1,  2, -1,   -1, -1,  6],

    [-1,  6, -1,   -1, -1, -1,    2,  8, -1],
    [-1, -1, -1,    4,  1,  9,   -1, -1,  5],
    [-1, -1, -1,   -1,  8, -1,   -1,  7,  9]
]


#aynı satırda 1 tane o sayıdan olacak
# aynı sütunda 1 tane o sayıdan olacak
# içinde bulunduğu 3x3 gridde 1 tane o sayıda olacak ve oyun 9 tane 3x3 grid içerecek
def is_valid(grid:list, num:int,r:int,c:int):
    row_value=[x for x in grid[r]]
    col_value=[grid[x][c] for x in range(0,9) ]
    #row da var mı
    if num in row_value:
        return False
    # col da var mı
    if num in col_value:
        return False
    #hangi 3x3 de olduğunun ataması

    row_normalized= r // 3  #mesela 5.satır 4.sütun yani index olarak [4,3] 4--> 1 3---> 1 
    col_normalized = c // 3 

    row_start= row_normalized * 3
    col_start = col_normalized *3

    # tüm 3x3 gridi dolaşıp checkle
    for i in range(row_start,row_start+3):
        for j in range(col_start,col_start+3):
            if grid[i][j]==num:
                return False
            
    
    return True
   
            
#böyle loop çok dönüyor son kaldığı yer gibi bişey olabilir
# en son kaldığı row en kaldığı col gibi bişey yapıp döngüyü rahatlatmaya çalışalım
def find_empty_cell(grid, last_row: int = 0, last_col: int = 0):
    # Aramaya en son kaldığımız satırdan (last_row) başlıyoruz
    for i in range(last_row, 9):
        
        # Eğer en son kaldığımız satırdaysak, sütuna 'last_col'dan başla.
        # Eğer alt satırlara geçtiysek, sütun aramasına en baştan (0'dan) başla.
        start_col = last_col if i == last_row else 0
        
        for j in range(start_col, 9):
            if grid[i][j] == -1:
                return i, j
                
    return None


def solve_sudoku(grid,last_row:int=0,last_col: int=0):
    
    empty_cell = find_empty_cell(grid,last_row,last_col)

    if  empty_cell==None:
        return True
    
    r,c = empty_cell
    for i in range(1,10):
        if is_valid(grid,i,r,c):
            grid[r][c]=i
            if  solve_sudoku(grid,r,c):
                return True
        
            # reset the number
            grid[r][c]=-1

    return False




In [31]:
def print_grid(grid):
    for i in range(9):
        # Her 3 satırda bir araya yatay ayırıcı çizgi çek (en üst hariç)
        if i % 3 == 0 and i != 0:
            print("- - - - - - - - - - - -")
            
        for j in range(9):
            # Her 3 sütunda bir araya dikey ayırıcı çizgi çek (en sol hariç)
            if j % 3 == 0 and j != 0:
                print(" | ", end="")
                
            # Satırın sonuna geldiysek alt satıra geçmek için normal print() kullan
            if j == 8:
                if grid[i][j] == -1:
                    print(".")
                else:
                    print(grid[i][j])
            else:
                # -1'leri nokta olarak yazdır, sayıları normal yazdır
                if grid[i][j] == -1:
                    print(". ", end="")
                else:
                    print(f"{grid[i][j]} ", end="")






In [28]:
solve_sudoku(sudoku_grid)
print_grid(sudoku_grid)
grid1=get_youdosudoku_grid("medium")


5 3 4  | 6 7 8  | 9 1 2
6 7 2  | 1 9 5  | 3 4 8
1 9 8  | 3 4 2  | 5 6 7
- - - - - - - - - - - -
8 5 9  | 7 6 1  | 4 2 3
4 2 6  | 8 5 3  | 7 9 1
7 1 3  | 9 2 4  | 8 5 6
- - - - - - - - - - - -
9 6 1  | 5 3 7  | 2 8 4
2 8 7  | 4 1 9  | 6 3 5
3 4 5  | 2 8 6  | 1 7 9
API'ye bağlanırken bir hata oluştu: 400 Client Error: Bad Request for url: https://www.youdosudoku.com/api
